# PredictMaint PY: Manutenção Preditiva com Machine Learning

**Contexto:** paradas não planejadas na linha de produção geram custo e atraso. A partir de 10.000 leituras de sensores (temperatura, rotação, torque e desgaste da ferramenta), este notebook constrói um pipeline que prevê se uma máquina vai falhar (`falha_maquina` = 1) ou operar normalmente (0).

**Objetivo:** comparar KNN e Árvore de Decisão e indicar o modelo de maior acurácia nos dados de teste.

## Configuração

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Ambiente pronto")

Ambiente pronto


   ## Fase 1: Análise Exploratória (EDA)

In [ ]:
df = pd.read_csv("data/manutencao_preditiva.csv")
print(f"A base tem {df.shape[0]} linhas e {df.shape[1]} colunas")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Valores nulos por coluna:")
print(df.isna().sum()[lambda s: s > 0])
print()
print("Linhas duplicadas:", df.duplicated().sum())
print("Valores repetidos em udi:", df["udi"].duplicated().sum())

In [ ]:
colunas_num = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min",
]

fig, eixos = plt.subplots(2, 3, figsize=(15, 8))
eixos = eixos.flatten()
for eixo, coluna in zip(eixos, colunas_num):
    sns.histplot(df[coluna], kde=True, ax=eixo)
    eixo.set_title(coluna)
eixos[-1].set_visible(False)
fig.suptitle("Distribuição das variáveis numéricas", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
contagem = df["falha_maquina"].value_counts().sort_index()
percentual = df["falha_maquina"].value_counts(normalize=True).sort_index() * 100

fig, eixo = plt.subplots(figsize=(6, 5))
sns.barplot(x=["Normal (0)", "Falha (1)"], y=contagem.values, ax=eixo)
for i, (qtd, pct) in enumerate(zip(contagem.values, percentual.values)):
    eixo.text(i, qtd, f"{qtd} ({pct:.2f}%)", ha="center", va="bottom")
eixo.set_title("Distribuição da variável alvo: falha_maquina")
eixo.set_ylabel("Quantidade de máquinas")
plt.show()

print(f"Acurácia ingênua (sempre prever 0): {percentual.loc[0]:.2f}%")

In [ ]:
correlacao = df[colunas_num + ["falha_maquina"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação de Pearson entre variáveis")
plt.show()

### Análise exploratória

**Estrutura da base.** O dataset tem 10.000 linhas e 14 colunas, sem nenhum registro duplicado. Os tipos são adequados: `tipo` e `id_produto` são texto, os sensores são numéricos e as colunas de falha são binárias (0 ou 1). Quatro colunas de sensores (`temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`) têm 500 valores nulos cada, 5% da base, sempre nas mesmas linhas. Esse ponto será tratado na Fase 2.

**Distribuições.** As variáveis têm escalas muito diferentes: a rotação gira em torno de 1.539 rpm, o torque em torno de 40 Nm e as temperaturas em torno de 300 K. A rotação é a única com distribuição claramente assimétrica, com cauda à direita e máximo de 2.886 rpm. Temperaturas, torque e desgaste têm formato próximo ao simétrico.

**Variável alvo.** Há forte desbalanceamento: apenas 339 máquinas (3,39%) falharam. Um modelo que sempre prevê "sem falha" acertaria 96,61% dos casos sem detectar nenhuma quebra. Essa acurácia ingênua será a referência de comparação na Fase 7.

**Correlações.** As duas temperaturas são fortemente correlacionadas entre si (0,88), o que indica informação em parte redundante. Em relação à falha, o torque tem a correlação mais forte (0,19), seguido do desgaste da ferramenta (0,11). Todas são fracas, o que sugere relações não lineares e justifica testar modelos como KNN e Árvore de Decisão.

**Implicações para a modelagem.** O desbalanceamento exige divisão estratificada e SMOTE aplicado só no treino. As escalas diferentes exigem StandardScaler no KNN. A assimetria da rotação orienta o uso da mediana na imputação. A relação entre torque, rotação e falha motiva criar a variável `potencia`.

## Fase 2: Limpeza e Tratamento (Data Prep)

## Fase 3: Feature Engineering

## Fase 4: Divisão e Balanceamento

## Fase 5: Escalonamento (StandardScaler)

## Fase 6: Ajuste de Parâmetros e Overfitting

## Fase 7: Avaliação da Acurácia e Veredito Final

## Conclusão e Melhorias